# Многопризнаковая регрессия на данных Diabetes

В этом учебном наборе 442 наблюдения и 10 признаков пациентов. Целевое значение — количественная оценка прогрессирования диабета через год после исходного обследования, а не диагноз «диабет / нет диабета». [Описание набора в документации scikit-learn](https://scikit-learn.org/stable/datasets/real_world.html).

В предыдущем примере `y = 2x + 1` сеть училась находить простую зависимость по одному входу. Здесь по десяти признакам нужно предсказать одно числовое значение, а зависимость не задана заранее одной формулой. Это многопризнаковая регрессия на данных пациентов.

После обучения оценим сеть на отложенных примерах и сравним её с простым прогнозом: средним целевым значением обучающей выборки.


In [235]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from open_neuron.activations import linear, relu, gelu, prelu, elu
from open_neuron.layers import Dropout, Layer
from open_neuron.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from open_neuron.models import NeuralNetwork
from open_neuron.optimizers import Adam, RMSprop, SGD

np.set_printoptions(precision=3, suppress=True)
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target.reshape(-1, 1)

print("Признаки:", len(diabetes.feature_names))
print("Пример имён признаков:", diabetes.feature_names[:3])
print("Размеры X и y:", X.shape, y.shape)
print("Диапазон целевых значений:", (y.min(), y.max()))

Признаки: 10
Пример имён признаков: ['age', 'sex', 'bmi']
Размеры X и y: (442, 10) (442, 1)
Диапазон целевых значений: (np.float64(25.0), np.float64(346.0))


## Разделяем и масштабируем данные

Сначала оставляем 20% примеров для итоговой проверки, а остальные используем для обучения. В начале ячейки создаётся свежий `split_seed`, поэтому каждый полный `Run All` получает новое разбиение. Этот один seed передаётся в `train_test_split` и используется всеми тремя моделями.

`StandardScaler` для каждого признака вычитает среднее и делит на стандартное отклонение. Он находит эти величины только по `X_train`; к тестовым данным применяет те же найденные параметры. Поэтому средние обучающих признаков после преобразования близки к нулю, но сами значения признаков не становятся нулевыми.

Целевые значения `y` не масштабируем: сеть предсказывает их в исходной шкале, и тестовые MAE/MSE тоже остаются в этой шкале.


In [236]:
split_seed = int(np.random.default_rng().integers(0, 2**32 - 1))
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=split_seed,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Seed разбиения:", split_seed)
print("Обучающих примеров:", len(X_train))
print("Тестовых примеров:  ", len(X_test))
print("Средние признаков train после масштабирования:",
      X_train_scaled.mean(axis=0))


Обучающих примеров: 353
Тестовых примеров:   89
Средние признаков train после масштабирования: [ 0.  0.  0.  0.  0.  0.  0.  0. -0.  0.]


## Создаём и обучаем сеть

Текущая конфигурация сети:

1. На вход поступают 10 признаков.
2. `Layer(10, activation=relu)` — полносвязный скрытый слой из 10 нейронов. Каждый нейрон вычисляет взвешенную сумму входов со смещением, затем ReLU заменяет отрицательный результат нулём.
3. `Dropout(0.1)` во время обучения случайно обнуляет 10% выходов предыдущего слоя. При `predict` Dropout выключен.
4. `Layer(1, activation=linear)` выдаёт одно вещественное число без ограничения диапазона.

Сеть обучается уменьшать MSE — среднее квадратов разности между предсказаниями и ответами. Adam использует шаг обучения `0.01` и clipping с `clip_norm=1.0`. В этой ячейке сеть обучается 250 эпох: примеры перед каждой эпохой перемешиваются, размер пакета постепенно растёт от 2 до 64. Перед созданием каждой модели устанавливается `np.random.seed(7)`, поэтому начальные веса одинаковы у Adam, SGD и RMSprop. При следующем полном запуске создаётся новый `split_seed`, а внутри этого запуска split и веса остаются общими.

В `fit` не передан отдельный `validation_data`, поэтому строка `Validation Loss ... on Train Data` означает потерю, пересчитанную после эпохи на всей обучающей выборке. Это не независимая проверка. `Loss` — среднее потерь обучающих мини-пакетов, посчитанных во время обновления весов; при этих проходах Dropout активен. Поэтому две величины могут различаться.

Тестовые примеры в `fit` не передаются. Они используются ниже для оценки после обучения.


In [237]:
np.random.seed(7)
model = NeuralNetwork(
    [
        Layer(10, activation=relu),
        Dropout(0.1),
        Layer(1, activation=linear),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="mse",
    optimizer=Adam(learning_rate=0.01, clip_norm=1.0),
)

model.fit(
    X_train_scaled,
    y_train,
    epochs=250,
    batch_size=2,
    final_batch_size=32,
    shuffle=True,
)

Training started with Overall Loss 30431.4837 on Train Data
Epoch 1/250, Loss: 27266.0786, Validation Loss: 22148.4806 on Train Data
Epoch 25/250, Loss: 3152.2250, Validation Loss: 2726.6958 on Train Data
Epoch 50/250, Loss: 2987.9869, Validation Loss: 2625.1889 on Train Data
Epoch 75/250, Loss: 2949.6275, Validation Loss: 2600.5347 on Train Data
Epoch 100/250, Loss: 3304.3443, Validation Loss: 2511.6122 on Train Data
Epoch 125/250, Loss: 2812.0106, Validation Loss: 2495.3921 on Train Data
Epoch 150/250, Loss: 3126.6783, Validation Loss: 2459.4422 on Train Data
Epoch 175/250, Loss: 2831.0139, Validation Loss: 2452.9988 on Train Data
Epoch 200/250, Loss: 3099.2542, Validation Loss: 2442.1537 on Train Data
Epoch 225/250, Loss: 2998.4051, Validation Loss: 2442.3024 on Train Data
Epoch 250/250, Loss: 2909.1785, Validation Loss: 2424.7564 on Train Data
Training completed with Overall Loss 2424.7564


## Повторяем обучение с SGD

Теперь оставляем те же данные, архитектуру, функцию потерь, размеры пакетов и число эпох, но заменяем Adam на SGD с `learning_rate=0.0001` и `momentum=0.9`. Перед созданием модели снова устанавливаем `np.random.seed(7)`, поэтому начальные веса, порядок перемешивания и маски Dropout совпадают с запуском Adam. Это делает сравнение оптимизаторов нагляднее: меняется только алгоритм обновления параметров.


In [238]:
np.random.seed(7)
sgd_model = NeuralNetwork(
    [
        Layer(10, activation=relu),
        Dropout(0.1),
        Layer(1, activation=linear),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="mse",
    optimizer=SGD(learning_rate=0.01, momentum=0.9, clip_norm=1.0),
)

sgd_model.fit(
    X_train_scaled,
    y_train,
    epochs=250,
    batch_size=2,
    final_batch_size=32,
    shuffle=True,
)

Training started with Overall Loss 30183.7752 on Train Data
Epoch 1/250, Loss: 16448.0313, Validation Loss: 3417.7913 on Train Data
Epoch 25/250, Loss: 2854.8924, Validation Loss: 2764.4187 on Train Data
Epoch 50/250, Loss: 2879.7885, Validation Loss: 2576.7328 on Train Data
Epoch 75/250, Loss: 2681.5265, Validation Loss: 2497.6838 on Train Data
Epoch 100/250, Loss: 2696.5924, Validation Loss: 2483.1150 on Train Data
Epoch 125/250, Loss: 2681.3169, Validation Loss: 2483.7043 on Train Data
Epoch 150/250, Loss: 2711.6838, Validation Loss: 2476.3133 on Train Data
Epoch 175/250, Loss: 2769.6731, Validation Loss: 2443.9099 on Train Data
Epoch 200/250, Loss: 2798.9250, Validation Loss: 2439.8329 on Train Data
Epoch 225/250, Loss: 2824.0361, Validation Loss: 2440.5195 on Train Data
Epoch 250/250, Loss: 2754.6846, Validation Loss: 2431.7628 on Train Data
Training completed with Overall Loss 2431.7628


## Повторяем обучение с RMSprop

Наконец, используем RMSprop с теми же условиями и `learning_rate=0.01`. RMSprop адаптирует шаг по скользящему среднему квадратов градиентов, но всё равно требует отдельного подбора скорости обучения.


In [239]:
np.random.seed(7)
rmsprop_model = NeuralNetwork(
    [
        Layer(10, activation=relu),
        Dropout(0.1),
        Layer(1, activation=linear),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="mse",
    optimizer=RMSprop(learning_rate=0.01, clip_norm=1.0),
)

rmsprop_model.fit(
    X_train_scaled,
    y_train,
    epochs=250,
    batch_size=2,
    final_batch_size=32,
    shuffle=True,
)

Training started with Overall Loss 29969.3754 on Train Data
Epoch 1/250, Loss: 25329.3407, Validation Loss: 17894.7066 on Train Data
Epoch 25/250, Loss: 3088.3545, Validation Loss: 2735.9881 on Train Data
Epoch 50/250, Loss: 3018.3254, Validation Loss: 2596.9071 on Train Data
Epoch 75/250, Loss: 3045.9972, Validation Loss: 2548.0560 on Train Data
Epoch 100/250, Loss: 3044.9934, Validation Loss: 2511.7904 on Train Data
Epoch 125/250, Loss: 2961.0830, Validation Loss: 2495.4764 on Train Data
Epoch 150/250, Loss: 3324.0105, Validation Loss: 2487.2587 on Train Data
Epoch 175/250, Loss: 3097.0797, Validation Loss: 2469.9600 on Train Data
Epoch 200/250, Loss: 2936.8905, Validation Loss: 2448.6827 on Train Data
Epoch 225/250, Loss: 3082.0629, Validation Loss: 2440.0755 on Train Data
Epoch 250/250, Loss: 2829.9469, Validation Loss: 2425.2319 on Train Data
Training completed with Overall Loss 2425.2319


## Оцениваем качество на тестовой выборке

После обучения `predict` получает масштабированные признаки `X_test_scaled` и выдаёт прогноз для каждого тестового примера. Сеть при этом работает в режиме предсказания: Dropout выключен.

- **MSE** — средний квадрат ошибки. Крупные промахи сильнее увеличивают эту метрику; её единицы — квадрат единиц целевого значения.
- **MAE** — средняя абсолютная ошибка в исходных единицах целевого значения; её проще читать как средний размер промаха.

Для сравнения базовый прогноз всегда равен среднему `y_train`. Он не использует признаки, поэтому показывает, насколько сеть полезнее постоянного ответа на этом разбиении.


In [244]:
predictions = model.predict(X_test_scaled)
baseline_predictions = np.full_like(y_test, y_train.mean())

mse = mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
baseline_mse = mean_squared_error(y_test, baseline_predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_r2 = r2_score(y_test, baseline_predictions)

print(f"Тестовая MSE Adam:       {mse:.2f}")
print(f"Тестовая MAE Adam:         {mae:.2f}")
print(f"Тестовый R² Adam:          {r2:.3f}")
print(f"Тестовая MSE среднего:   {baseline_mse:.2f}")
print(f"Тестовая MAE среднего:     {baseline_mae:.2f}")
print(f"Тестовый R² среднего:      {baseline_r2:.3f}")
print()
print("Первые предсказания Adam и ответы:")
for expected, predicted in zip(y_test.ravel()[:5], predictions.ravel()[:5]):
  print(f"ответ: {expected:.1f}, предсказание: {predicted:.1f}")

Тестовая MSE Adam:       3059.88
Тестовая MAE Adam:         43.19
Тестовый R² Adam:          0.407
Тестовая MSE среднего:   5255.88
Тестовая MAE среднего:     61.43
Тестовый R² среднего:      -0.019

Первые предсказания Adam и ответы:
ответ: 139.0, предсказание: 208.2
ответ: 179.0, предсказание: 151.0
ответ: 104.0, предсказание: 92.1
ответ: 128.0, предсказание: 165.1
ответ: 67.0, предсказание: 194.9


In [241]:
sgd_predictions = sgd_model.predict(X_test_scaled)
sgd_mse = mean_squared_error(y_test, sgd_predictions)
sgd_mae = mean_absolute_error(y_test, sgd_predictions)
sgd_r2 = r2_score(y_test, sgd_predictions)

print(f"Тестовая MSE SGD:        {sgd_mse:.2f}")
print(f"Тестовая MAE SGD:          {sgd_mae:.2f}")
print(f"Тестовый R² SGD:           {sgd_r2:.3f}")
print(f"Тестовая MSE среднего:   {baseline_mse:.2f}")
print(f"Тестовая MAE среднего:     {baseline_mae:.2f}")
print(f"Тестовый R² среднего:      {baseline_r2:.3f}")
print()
print("Первые предсказания SGD и ответы:")
for expected, predicted in zip(y_test.ravel()[:5], sgd_predictions.ravel()[:5]):
  print(f"ответ: {expected:.1f}, предсказание: {predicted:.1f}")

Тестовая MSE SGD:        3277.07
Тестовая MAE SGD:          45.16
Тестовый R² SGD:           0.365
Тестовая MSE среднего:   5255.88
Тестовая MAE среднего:     61.43
Тестовый R² среднего:      -0.019

Первые предсказания SGD и ответы:
ответ: 139.0, предсказание: 212.3
ответ: 179.0, предсказание: 150.5
ответ: 104.0, предсказание: 105.9
ответ: 128.0, предсказание: 159.2
ответ: 67.0, предсказание: 196.9


In [242]:
rmsprop_predictions = rmsprop_model.predict(X_test_scaled)
rmsprop_mse = mean_squared_error(y_test, rmsprop_predictions)
rmsprop_mae = mean_absolute_error(y_test, rmsprop_predictions)
rmsprop_r2 = r2_score(y_test, rmsprop_predictions)

print(f"Тестовая MSE RMSprop:    {rmsprop_mse:.2f}")
print(f"Тестовая MAE RMSprop:      {rmsprop_mae:.2f}")
print(f"Тестовый R² RMSprop:       {rmsprop_r2:.3f}")
print(f"Тестовая MSE среднего:   {baseline_mse:.2f}")
print(f"Тестовая MAE среднего:     {baseline_mae:.2f}")
print(f"Тестовый R² среднего:      {baseline_r2:.3f}")
print()
print("Первые предсказания RMSprop и ответы:")
for expected, predicted in zip(y_test.ravel()[:5], rmsprop_predictions.ravel()[:5]):
  print(f"ответ: {expected:.1f}, предсказание: {predicted:.1f}")

Тестовая MSE RMSprop:    3187.36
Тестовая MAE RMSprop:      45.01
Тестовый R² RMSprop:       0.382
Тестовая MSE среднего:   5255.88
Тестовая MAE среднего:     61.43
Тестовый R² среднего:      -0.019

Первые предсказания RMSprop и ответы:
ответ: 139.0, предсказание: 203.4
ответ: 179.0, предсказание: 150.3
ответ: 104.0, предсказание: 101.5
ответ: 128.0, предсказание: 159.2
ответ: 67.0, предсказание: 195.3


## Результаты трёх запусков

Все три модели используют одну архитектуру из 20 ReLU-нейронов, Dropout `0.2`, линейный выход и MSE. В текущем коде включён clipping `clip_norm=1.0`; для Adam задан `learning_rate=0.01`, для SGD — `learning_rate=0.01` и `momentum=0.9`, для RMSprop — `learning_rate=0.01`. Размер пакета меняется от 2 до 64 в течение 250 эпох.

В начале каждого полного запуска печатается новый `split_seed`. Он один для всех трёх моделей; перед каждой моделью seed `7` выравнивает начальные веса. Поэтому следующий `Run All` получает новый split, а Adam, SGD и RMSprop внутри одного запуска проверяются на одинаковых данных.

Ноутбук печатает для каждой модели MSE, MAE и R², а также базовый прогноз средним. MSE сильнее учитывает крупные промахи, MAE показывает средний абсолютный промах в исходных единицах, а R² сравнивает модель с прогнозом средним: у baseline он равен `0` при расчёте на той же выборке.

Первые пять предсказаний показывают отдельные случаи, а не заменяют общие метрики. Сравнение относится к одному конкретному запуску и выбранному разбиению данных.


In [243]:
model.summary()

Info:
Network type: "Regression", layers: 2, neurons: 11, optimizer: "Adam", epochs: 250, batch size: 2-32, loss function: "Mean Squared Error"
Last forward:
Layer 1, neurons: 10, inputs: [-0.881 -0.926 -0.684 -0.476  1.041  0.947  1.124 -0.491 -0.526 -0.784], activation: relu, output (a): [3.551 5.858 0.    4.878 0.    2.118 0.    6.591 0.    0.   ]
Unit 1/10, inputs: [-0.881 -0.926 -0.684 -0.476  1.041  0.947  1.124 -0.491 -0.526 -0.784], weights: [-1.146 -1.036  0.775 -0.268 -2.099  2.492 -1.687  0.507  1.673 -0.578], bias: [4.382], z: 3.5508167255802996, activation: relu, a: 3.5508167255802996
Unit 2/10, inputs: [-0.881 -0.926 -0.684 -0.476  1.041  0.947  1.124 -0.491 -0.526 -0.784], weights: [-0.891 -1.242  0.246  0.281 -1.798  1.751  0.136  1.059  1.965 -0.517], bias: [5.437], z: 5.857936392841025, activation: relu, a: 5.857936392841025
...
Unit 9/10, inputs: [-0.881 -0.926 -0.684 -0.476  1.041  0.947  1.124 -0.491 -0.526 -0.784], weights: [ 0.656  1.294  2.77   2.895 -0.725  1.2